In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
GSPINN Project 1: one-file Kaggle runner
=========================================

Purpose
-------
This single file trains lightweight GSPINN versions of the four SISC benchmarks
and computes the residual-to-error certification quantities needed for the new
manuscript section:

    eta_val          = sum of validation residuals + matching defects + boundary defects,
    E_ref            = segmentwise reference error when a reference is available,
    E_ref/(eps+eta) = empirical stability ratio.

How to use in Kaggle
--------------------
1. Upload this file to a Kaggle notebook.
2. Run all cells if imported as a notebook, or run:
       !python GSPINN_Project1_Kaggle_OneFile.py
3. Outputs are written to:
       /kaggle/working/gspinn_project1_outputs
   or, outside Kaggle, to:
       ./gspinn_project1_outputs

Important
---------
The default RUN_MODE is "pilot" so the file finishes quickly and checks that the
pipeline works.  For paper-quality runs, change only one line below:

    RUN_MODE = "paper"

Paper mode is more expensive.  It uses a wider epsilon grid and more epochs.
The PNP benchmark is treated as residual/matching certification unless a trusted
reference trajectory is supplied later.
"""

from __future__ import annotations
import os
import math
import json
import time
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from scipy.integrate import solve_ivp

# =============================================================================
# 0. Configuration: this is the only place you may want to edit later.
# =============================================================================

RUN_MODE = "paper"     # "pilot" for quick test; "paper" for final-quality runs.
RANDOM_SEED = 1234
BENCHMARKS = ["reaction_diffusion", "lorenz", "nonhyperbolic", "pnp"]

# If True, the code saves model checkpoints.  Leave False for the first Kaggle run.
SAVE_MODELS = False

# Use double precision to stay close to the original SISC notebooks.
DTYPE = torch.float64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if RUN_MODE == "pilot":
    EPS_VALUES = [1.0e-2]
    N_POINTS = 80
    EPOCHS = {
        "reaction_diffusion": 80,
        "lorenz": 80,
        "nonhyperbolic": 80,
        "pnp": 80,
    }
    PRINT_EVERY = 20
elif RUN_MODE == "paper":
    EPS_VALUES = list(np.logspace(-5, -1, 12))
    N_POINTS = 250
    EPOCHS = {
        "reaction_diffusion": 20000,
        "lorenz": 30000,
        "nonhyperbolic": 30000,
        "pnp": 50000,
    }
    PRINT_EVERY = 2000
else:
    raise ValueError("RUN_MODE must be 'pilot' or 'paper'.")

try:
    BASE_DIR = Path("/kaggle/working")
    if not BASE_DIR.exists():
        BASE_DIR = Path.cwd()
except Exception:
    BASE_DIR = Path.cwd()

OUT_DIR = BASE_DIR / "gspinn_project1_outputs"
FIG_DIR = OUT_DIR / "figures"
CSV_DIR = OUT_DIR / "csv"
MODEL_DIR = OUT_DIR / "models"
for d in [OUT_DIR, FIG_DIR, CSV_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =============================================================================
# 1. Reproducibility and general utilities
# =============================================================================

def set_seed(seed: int = 1234) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)

def to_tensor(x: np.ndarray) -> torch.Tensor:
    return torch.tensor(x, dtype=DTYPE, device=DEVICE)


def detach_np(x: torch.Tensor) -> np.ndarray:
    return x.detach().cpu().numpy()


def grad(outputs: torch.Tensor, inputs: torch.Tensor) -> torch.Tensor:
    return torch.autograd.grad(
        outputs.sum(), inputs, retain_graph=True, create_graph=True
    )[0]


def norm2_rows(a: np.ndarray) -> np.ndarray:
    a = np.asarray(a, dtype=float)
    if a.ndim == 1:
        a = a.reshape(-1, 1)
    return np.linalg.norm(a, axis=1)


def trapz_l1(residual: np.ndarray, time_grid: np.ndarray) -> float:
    xgrid = time_grid.reshape(-1)
    yvals = norm2_rows(residual)
    area = np.trapezoid(yvals, xgrid) if hasattr(np, "trapezoid") else np.trapz(yvals, xgrid)
    return float(abs(area))


def trapz_l2(residual: np.ndarray, time_grid: np.ndarray) -> float:
    xgrid = time_grid.reshape(-1)
    yvals = np.sum(residual**2, axis=1)
    area = np.trapezoid(yvals, xgrid) if hasattr(np, "trapezoid") else np.trapz(yvals, xgrid)
    return float(np.sqrt(max(abs(area), 0.0)))


def linf(residual: np.ndarray) -> float:
    return float(np.max(norm2_rows(residual)))


def endpoint_defect(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(np.asarray(a).reshape(-1) - np.asarray(b).reshape(-1)))


class SegmentMLP(nn.Module):
    """Small tanh network with built-in affine time normalization."""

    def __init__(self, dim_out: int, hidden: int, t0: float, t1: float):
        super().__init__()
        self.dim_out = dim_out
        self.hidden = hidden
        self.t0 = float(t0)
        self.t1 = float(t1)
        self.net = nn.Sequential(
            nn.Linear(1, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, dim_out),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # Normalize to roughly [-1,1]. Autograd keeps the correct chain rule.
        z = 2.0 * (t - self.t0) / (self.t1 - self.t0) - 1.0
        return self.net(z)


@dataclass
class SegmentResult:
    name: str
    time: np.ndarray
    pred: np.ndarray
    residual: np.ndarray
    exact: Optional[np.ndarray]

    @property
    def l1_residual(self) -> float:
        return trapz_l1(self.residual, self.time)

    @property
    def l2_residual(self) -> float:
        return trapz_l2(self.residual, self.time)

    @property
    def linf_residual(self) -> float:
        return linf(self.residual)

    @property
    def linf_error(self) -> float:
        if self.exact is None:
            return float("nan")
        return linf(self.pred - self.exact)

    @property
    def l2_error(self) -> float:
        if self.exact is None:
            return float("nan")
        return trapz_l2(self.pred - self.exact, self.time)


@dataclass
class RunCertificate:
    benchmark: str
    eps: float
    eta_l1: float
    eta_l2_components: float
    error_linf_sum: float
    error_l2_components: float
    matching_defect: float
    boundary_defect: float
    stability_ratio_eps_eta: float
    stability_ratio_eta: float
    final_loss: float
    epochs: int
    n_points: int
    runtime_seconds: float


def integrate_reference(rhs: Callable[[float, np.ndarray], np.ndarray],
                        time_grid: np.ndarray,
                        y0: np.ndarray,
                        method: str = "Radau") -> Optional[np.ndarray]:
    """Integrate reference ODE on the same time grid.

    If the IVP fails or becomes numerically unreasonable, return None.  This is
    safer than reporting misleading reference errors.
    """
    time_grid = np.asarray(time_grid, dtype=float).reshape(-1)
    y0 = np.asarray(y0, dtype=float).reshape(-1)
    if len(time_grid) < 2:
        return None
    try:
        sol = solve_ivp(
            fun=lambda t, y: rhs(t, y),
            t_span=(float(time_grid[0]), float(time_grid[-1])),
            y0=y0,
            t_eval=time_grid,
            method=method,
            rtol=1e-8,
            atol=1e-10,
        )
        if (not sol.success) or sol.y.shape[1] != len(time_grid):
            return None
        arr = sol.y.T
        if not np.all(np.isfinite(arr)):
            return None
        if np.max(np.abs(arr)) > 1e8:
            return None
        return arr
    except Exception:
        return None


def compute_certificate(benchmark: str,
                        eps: float,
                        segments: List[SegmentResult],
                        matching_defects: List[float],
                        boundary_defects: List[float],
                        final_loss: float,
                        epochs: int,
                        n_points: int,
                        runtime_seconds: float) -> RunCertificate:
    eta_l1 = float(sum(s.l1_residual for s in segments) + sum(matching_defects) + sum(boundary_defects))
    eta_l2 = float(math.sqrt(sum(s.l2_residual**2 for s in segments)) + sum(matching_defects) + sum(boundary_defects))
    finite_linf_errors = [s.linf_error for s in segments if np.isfinite(s.linf_error)]
    finite_l2_errors = [s.l2_error for s in segments if np.isfinite(s.l2_error)]
    if len(finite_linf_errors) == 0:
        error_linf_sum = float("nan")
        error_l2_components = float("nan")
        ratio_eps_eta = float("nan")
        ratio_eta = float("nan")
    else:
        error_linf_sum = float(sum(finite_linf_errors))
        error_l2_components = float(math.sqrt(sum(e**2 for e in finite_l2_errors)))
        ratio_eps_eta = float(error_linf_sum / (eps + eta_l1)) if eps + eta_l1 > 0 else float("nan")
        ratio_eta = float(error_linf_sum / eta_l1) if eta_l1 > 0 else float("nan")
    return RunCertificate(
        benchmark=benchmark,
        eps=float(eps),
        eta_l1=eta_l1,
        eta_l2_components=eta_l2,
        error_linf_sum=error_linf_sum,
        error_l2_components=error_l2_components,
        matching_defect=float(sum(matching_defects)),
        boundary_defect=float(sum(boundary_defects)),
        stability_ratio_eps_eta=ratio_eps_eta,
        stability_ratio_eta=ratio_eta,
        final_loss=float(final_loss),
        epochs=int(epochs),
        n_points=int(n_points),
        runtime_seconds=float(runtime_seconds),
    )


def save_run_outputs(benchmark: str,
                     eps: float,
                     segments: List[SegmentResult],
                     certificate: RunCertificate,
                     loss_history: List[float]) -> None:
    eps_tag = f"{eps:.0e}".replace("-", "m").replace("+", "p")
    npz_path = OUT_DIR / f"{benchmark}_eps_{eps_tag}_trajectories.npz"
    payload = {"eps": eps, "loss_history": np.asarray(loss_history, dtype=float)}
    for s in segments:
        payload[f"time_{s.name}"] = s.time
        payload[f"pred_{s.name}"] = s.pred
        payload[f"residual_{s.name}"] = s.residual
        if s.exact is not None:
            payload[f"exact_{s.name}"] = s.exact
    np.savez(npz_path, **payload)

    json_path = OUT_DIR / f"{benchmark}_eps_{eps_tag}_certificate.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(asdict(certificate), f, indent=2)

    # Loss plot.
    plt.figure(figsize=(6.2, 4.2))
    y = np.asarray(loss_history, dtype=float)
    y = np.maximum(y, 1e-300)
    plt.semilogy(np.arange(len(y)), y)
    plt.xlabel("epoch")
    plt.ylabel("training loss")
    plt.title(f"{benchmark.replace('_',' ')}: eps={eps:.1e}")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{benchmark}_eps_{eps_tag}_loss.png", dpi=220)
    plt.close()

    # Phase-space / state plot.
    plt.figure(figsize=(6.2, 4.2))
    for s in segments:
        if s.pred.shape[1] >= 2:
            plt.plot(s.pred[:, 0], s.pred[:, 1], label=s.name)
    plt.xlabel("component 1")
    plt.ylabel("component 2")
    plt.title(f"{benchmark.replace('_',' ')} segmented trajectory")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{benchmark}_eps_{eps_tag}_trajectory.png", dpi=220)
    plt.close()

# =============================================================================
# 2. Benchmark-specific residuals and training routines
# =============================================================================

def train_reaction_diffusion(eps: float, epochs: int, n_points: int) -> Tuple[List[SegmentResult], RunCertificate, List[float]]:
    name = "reaction_diffusion"
    start = time.time()
    hidden = 16
    phys_w = 10.0
    match_w = 50.0
    bc_w = 50.0

    t = np.linspace(0.0, 70.0, n_points)
    tau = np.linspace(0.0, 0.70, n_points)
    t_t = to_tensor(t.reshape(-1, 1)); t_t.requires_grad_(True)
    tau_t = to_tensor(tau.reshape(-1, 1)); tau_t.requires_grad_(True)

    left = to_tensor(np.array([1.0, 2.0]))
    right = to_tensor(np.array([2.0, 2.0]))

    mf = SegmentMLP(2, hidden, t[0], t[-1]).to(device=DEVICE, dtype=DTYPE)
    ms = SegmentMLP(2, hidden, tau[0], tau[-1]).to(device=DEVICE, dtype=DTYPE)
    opt = torch.optim.Adam(list(mf.parameters()) + list(ms.parameters()), lr=1e-3)
    loss_history = []

    def residuals():
        pf = mf(t_t); ps = ms(tau_t)
        xf, yf = pf[:, 0:1], pf[:, 1:2]
        xs, ys = ps[:, 0:1], ps[:, 1:2]
        dxf = grad(xf, t_t); dyf = grad(yf, t_t)
        dxs = grad(xs, tau_t); dys = grad(ys, tau_t)
        rf = torch.cat([dxf - eps * yf, dyf - xf + yf], dim=1)
        rs = torch.cat([dxs - ys, eps * dys - xs + ys], dim=1)
        return pf, ps, rf, rs

    for ep in range(epochs + 1):
        opt.zero_grad()
        pf, ps, rf, rs = residuals()
        loss_phys = torch.mean(torch.sum(rf**2, dim=1)) + torch.mean(torch.sum(rs**2, dim=1))
        loss_bc = torch.sum((pf[0] - left)**2) + torch.sum((ps[-1] - right)**2)
        loss_match = torch.sum((pf[-1] - ps[0])**2)
        loss = phys_w * loss_phys + bc_w * loss_bc + match_w * loss_match
        loss.backward(); opt.step()
        loss_history.append(float(loss.detach().cpu()))
        if ep % PRINT_EVERY == 0 or ep == epochs:
            print(f"[{name} eps={eps:.1e}] epoch {ep:6d}/{epochs}, loss={loss_history[-1]:.3e}")

    pf, ps, rf, rs = residuals()
    pred_f = detach_np(pf); pred_s = detach_np(ps)
    res_f = detach_np(rf); res_s = detach_np(rs)

    def rhs_fast(tt, y):
        return np.array([eps * y[1], y[0] - y[1]], dtype=float)
    def rhs_slow(tt, y):
        return np.array([y[1], (y[0] - y[1]) / eps], dtype=float)

    exact_f = integrate_reference(rhs_fast, t, pred_f[0], method="Radau")
    exact_s = integrate_reference(rhs_slow, tau, pred_s[0], method="Radau")
    segs = [
        SegmentResult("fast", t, pred_f, res_f, exact_f),
        SegmentResult("slow", tau, pred_s, res_s, exact_s),
    ]
    matching = [endpoint_defect(pred_f[-1], pred_s[0])]
    boundary = [endpoint_defect(pred_f[0], np.array([1.0, 2.0])), endpoint_defect(pred_s[-1], np.array([2.0, 2.0]))]
    cert = compute_certificate(name, eps, segs, matching, boundary, loss_history[-1], epochs, n_points, time.time() - start)
    if SAVE_MODELS:
        torch.save({"fast": mf.state_dict(), "slow": ms.state_dict()}, MODEL_DIR / f"{name}_eps_{eps:.0e}.pt")
    return segs, cert, loss_history


def train_lorenz(eps: float, epochs: int, n_points: int) -> Tuple[List[SegmentResult], RunCertificate, List[float]]:
    name = "lorenz"
    start_time = time.time()
    hidden = 16
    phys_w = 5.0
    match_w = 50.0
    bc_w = 50.0

    t = np.linspace(0.0, 30.0, n_points)
    tau = np.linspace(0.0, 1.0, n_points)
    tb = np.linspace(0.0, -30.0, n_points)
    t_t = to_tensor(t.reshape(-1, 1)); t_t.requires_grad_(True)
    tau_t = to_tensor(tau.reshape(-1, 1)); tau_t.requires_grad_(True)
    tb_t = to_tensor(tb.reshape(-1, 1)); tb_t.requires_grad_(True)

    left = to_tensor(np.array([1.0, 0.0, 0.0]))
    right = to_tensor(np.array([0.0, 1.0, 1.0]))
    mf = SegmentMLP(3, hidden, t[0], t[-1]).to(device=DEVICE, dtype=DTYPE)
    ms = SegmentMLP(3, hidden, tau[0], tau[-1]).to(device=DEVICE, dtype=DTYPE)
    mb = SegmentMLP(3, hidden, tb[0], tb[-1]).to(device=DEVICE, dtype=DTYPE)
    opt = torch.optim.Adam(list(mf.parameters()) + list(ms.parameters()) + list(mb.parameters()), lr=1e-3)
    loss_history = []

    def residuals():
        pf = mf(t_t); ps = ms(tau_t); pb = mb(tb_t)
        xf, yf, zf = pf[:, 0:1], pf[:, 1:2], pf[:, 2:3]
        xs, ys, zs = ps[:, 0:1], ps[:, 1:2], ps[:, 2:3]
        xb, yb, zb = pb[:, 0:1], pb[:, 1:2], pb[:, 2:3]
        r_f = torch.cat([
            grad(xf, t_t) + xf,
            grad(yf, t_t) - 2.0*yf - eps*xf,
            grad(zf, t_t) - eps*(xf**2 + 1.0),
        ], dim=1)
        # Reduced slow flow on x=y=0, z'=1, with a soft manifold penalty.
        r_s = torch.cat([
            grad(xs, tau_t),
            grad(ys, tau_t),
            grad(zs, tau_t) - 1.0,
            xs,
            ys,
        ], dim=1)
        r_b = torch.cat([
            grad(xb, tb_t) + xb,
            grad(yb, tb_t) - 2.0*yb - eps*xb,
            grad(zb, tb_t) - eps*(xb**2 + 1.0),
        ], dim=1)
        return pf, ps, pb, r_f, r_s, r_b

    for ep in range(epochs + 1):
        opt.zero_grad()
        pf, ps, pb, r_f, r_s, r_b = residuals()
        loss_phys = torch.mean(torch.sum(r_f**2, dim=1)) + torch.mean(torch.sum(r_s**2, dim=1)) + torch.mean(torch.sum(r_b**2, dim=1))
        loss_bc = torch.sum((pf[0] - left)**2) + torch.sum((pb[0] - right)**2)
        loss_match = torch.sum((pf[-1] - ps[0])**2) + torch.sum((ps[-1] - pb[-1])**2)
        loss = phys_w*loss_phys + bc_w*loss_bc + match_w*loss_match
        loss.backward(); opt.step()
        loss_history.append(float(loss.detach().cpu()))
        if ep % PRINT_EVERY == 0 or ep == epochs:
            print(f"[{name} eps={eps:.1e}] epoch {ep:6d}/{epochs}, loss={loss_history[-1]:.3e}")

    pf, ps, pb, r_f, r_s, r_b = residuals()
    pred_f = detach_np(pf); pred_s = detach_np(ps); pred_b = detach_np(pb)
    res_f = detach_np(r_f); res_s = detach_np(r_s[:, 0:3]); res_b = detach_np(r_b)

    def rhs_fast(tt, y):
        x, yy, z = y
        return np.array([-x, 2.0*yy + eps*x, eps*(x*x + 1.0)], dtype=float)
    def rhs_slow_reduced(tt, y):
        return np.array([0.0, 0.0, 1.0], dtype=float)

    exact_f = integrate_reference(rhs_fast, t, pred_f[0], method="Radau")
    exact_s = integrate_reference(rhs_slow_reduced, tau, pred_s[0], method="RK45")
    exact_b = integrate_reference(rhs_fast, tb, pred_b[0], method="Radau")
    segs = [
        SegmentResult("fast", t, pred_f, res_f, exact_f),
        SegmentResult("slow", tau, pred_s, res_s, exact_s),
        SegmentResult("fast2", tb, pred_b, res_b, exact_b),
    ]
    matching = [endpoint_defect(pred_f[-1], pred_s[0]), endpoint_defect(pred_s[-1], pred_b[-1])]
    boundary = [endpoint_defect(pred_f[0], np.array([1.0, 0.0, 0.0])), endpoint_defect(pred_b[0], np.array([0.0, 1.0, 1.0]))]
    cert = compute_certificate(name, eps, segs, matching, boundary, loss_history[-1], epochs, n_points, time.time() - start_time)
    if SAVE_MODELS:
        torch.save({"fast": mf.state_dict(), "slow": ms.state_dict(), "fast2": mb.state_dict()}, MODEL_DIR / f"{name}_eps_{eps:.0e}.pt")
    return segs, cert, loss_history


def train_nonhyperbolic(eps: float, epochs: int, n_points: int) -> Tuple[List[SegmentResult], RunCertificate, List[float]]:
    name = "nonhyperbolic"
    start_time = time.time()
    hidden = 18
    phys_w = 1.0
    match_w = 50.0
    bc_w = 50.0

    t = np.linspace(0.0, 10.0, n_points)
    tau = np.linspace(0.0, 4.8, n_points)
    tb = np.linspace(0.0, -10.0, n_points)
    t_t = to_tensor(t.reshape(-1, 1)); t_t.requires_grad_(True)
    tau_t = to_tensor(tau.reshape(-1, 1)); tau_t.requires_grad_(True)
    tb_t = to_tensor(tb.reshape(-1, 1)); tb_t.requires_grad_(True)

    left = to_tensor(np.array([-5.0, -5.0]))
    right = to_tensor(np.array([0.0, 2.0]))
    mf = SegmentMLP(2, hidden, t[0], t[-1]).to(device=DEVICE, dtype=DTYPE)
    ms = SegmentMLP(2, hidden, tau[0], tau[-1]).to(device=DEVICE, dtype=DTYPE)
    mb = SegmentMLP(2, hidden, tb[0], tb[-1]).to(device=DEVICE, dtype=DTYPE)
    opt = torch.optim.Adam(list(mf.parameters()) + list(ms.parameters()) + list(mb.parameters()), lr=1e-3)
    loss_history = []

    def residuals():
        pf = mf(t_t); ps = ms(tau_t); pb = mb(tb_t)
        xf, yf = pf[:, 0:1], pf[:, 1:2]
        xs, ys = ps[:, 0:1], ps[:, 1:2]
        xb, yb = pb[:, 0:1], pb[:, 1:2]
        r_f = torch.cat([grad(xf, t_t) - eps, grad(yf, t_t) - (xf + yf**2)], dim=1)
        r_s = torch.cat([grad(xs, tau_t) - 1.0, eps*grad(ys, tau_t) - (xs + ys**2)], dim=1)
        r_b = torch.cat([grad(xb, tb_t) - eps, grad(yb, tb_t) - (xb + yb**2)], dim=1)
        return pf, ps, pb, r_f, r_s, r_b

    for ep in range(epochs + 1):
        opt.zero_grad()
        pf, ps, pb, r_f, r_s, r_b = residuals()
        loss_phys = torch.mean(torch.sum(r_f**2, dim=1)) + torch.mean(torch.sum(r_s**2, dim=1)) + torch.mean(torch.sum(r_b**2, dim=1))
        loss_bc = torch.sum((pf[0] - left)**2) + torch.sum((pb[0] - right)**2)
        loss_match = torch.sum((pf[-1] - ps[0])**2) + torch.sum((ps[-1] - pb[-1])**2)
        loss = phys_w*loss_phys + bc_w*loss_bc + match_w*loss_match
        loss.backward(); opt.step()
        loss_history.append(float(loss.detach().cpu()))
        if ep % PRINT_EVERY == 0 or ep == epochs:
            print(f"[{name} eps={eps:.1e}] epoch {ep:6d}/{epochs}, loss={loss_history[-1]:.3e}")

    pf, ps, pb, r_f, r_s, r_b = residuals()
    pred_f = detach_np(pf); pred_s = detach_np(ps); pred_b = detach_np(pb)
    res_f = detach_np(r_f); res_s = detach_np(r_s); res_b = detach_np(r_b)

    def rhs_fast(tt, y):
        x, yy = y
        return np.array([eps, x + yy*yy], dtype=float)
    def rhs_slow(tt, y):
        x, yy = y
        return np.array([1.0, (x + yy*yy)/eps], dtype=float)

    # This model is a stress test near loss of normal hyperbolicity. Reference IVPs
    # can fail or become very sensitive. That is scientifically meaningful, so we
    # keep NaN if solve_ivp cannot return a reliable trajectory.
    exact_f = integrate_reference(rhs_fast, t, pred_f[0], method="Radau")
    exact_s = integrate_reference(rhs_slow, tau, pred_s[0], method="Radau")
    exact_b = integrate_reference(rhs_fast, tb, pred_b[0], method="Radau")
    segs = [
        SegmentResult("fast", t, pred_f, res_f, exact_f),
        SegmentResult("slow", tau, pred_s, res_s, exact_s),
        SegmentResult("fast2", tb, pred_b, res_b, exact_b),
    ]
    matching = [endpoint_defect(pred_f[-1], pred_s[0]), endpoint_defect(pred_s[-1], pred_b[-1])]
    boundary = [endpoint_defect(pred_f[0], np.array([-5.0, -5.0])), endpoint_defect(pred_b[0], np.array([0.0, 2.0]))]
    cert = compute_certificate(name, eps, segs, matching, boundary, loss_history[-1], epochs, n_points, time.time() - start_time)
    if SAVE_MODELS:
        torch.save({"fast": mf.state_dict(), "slow": ms.state_dict(), "fast2": mb.state_dict()}, MODEL_DIR / f"{name}_eps_{eps:.0e}.pt")
    return segs, cert, loss_history


def train_pnp(eps: float, epochs: int, n_points: int) -> Tuple[List[SegmentResult], RunCertificate, List[float]]:
    name = "pnp"
    start_time = time.time()
    hidden = 20
    phys_w = 2.0
    match_w = 50.0
    bc_w = 50.0
    z1, z2 = 1.0, -1.0
    V = -10.0
    l = 1.0
    l1, l2 = l + 0.1, l - 0.1
    r = 0.5
    r1, r2 = r - 0.1, r + 0.1

    t = np.linspace(0.0, 20.0, n_points)
    tau = np.linspace(0.0, 1.0, n_points)
    tb = np.linspace(0.0, -20.0, n_points)
    t_t = to_tensor(t.reshape(-1, 1)); t_t.requires_grad_(True)
    tau_t = to_tensor(tau.reshape(-1, 1)); tau_t.requires_grad_(True)
    tb_t = to_tensor(tb.reshape(-1, 1)); tb_t.requires_grad_(True)

    mf = SegmentMLP(7, hidden, t[0], t[-1]).to(device=DEVICE, dtype=DTYPE)
    ms = SegmentMLP(7, hidden, tau[0], tau[-1]).to(device=DEVICE, dtype=DTYPE)
    mb = SegmentMLP(7, hidden, tb[0], tb[-1]).to(device=DEVICE, dtype=DTYPE)
    opt = torch.optim.Adam(list(mf.parameters()) + list(ms.parameters()) + list(mb.parameters()), lr=1e-3)
    loss_history = []

    left_partial = to_tensor(np.array([l1, l2, 0.0]))       # c1,c2,w at x=0
    right_partial = to_tensor(np.array([r1, r2, 1.0]))      # c1,c2,w at x=1

    def fast_residual(p: torch.Tensor, tt: torch.Tensor) -> torch.Tensor:
        phi, u, c1, c2, j1, j2, w = [p[:, i:i+1] for i in range(7)]
        return torch.cat([
            grad(phi, tt) - u,
            grad(u, tt) + z1*c1 + z2*c2,
            grad(c1, tt) + z1*c1*u + eps*j1,
            grad(c2, tt) + z2*c2*u + eps*j2,
            grad(j1, tt),
            grad(j2, tt),
            grad(w, tt) - eps,
            torch.relu(-c1),
            torch.relu(-c2),
        ], dim=1)

    def slow_residual(p: torch.Tensor, tt: torch.Tensor) -> torch.Tensor:
        phi, u, c1, c2, j1, j2, w = [p[:, i:i+1] for i in range(7)]
        c1_safe = torch.clamp(c1, min=1e-4)
        denom = z1 * (z1 - z2) * c1_safe
        q = - (z1*j1 + z2*j2) / denom
        return torch.cat([
            u,
            grad(phi, tt) - q,
            grad(c1, tt) + z1*c1*q + j1,
            grad(c2, tt) + z2*c2*q + j2,
            grad(j1, tt),
            grad(j2, tt),
            grad(w, tt) - 1.0,
            z1*c1 + z2*c2,
            torch.relu(-c1),
            torch.relu(-c2),
        ], dim=1)

    for ep in range(epochs + 1):
        opt.zero_grad()
        pf = mf(t_t); ps = ms(tau_t); pb = mb(tb_t)
        r_f = fast_residual(pf, t_t)
        r_s = slow_residual(ps, tau_t)
        r_b = fast_residual(pb, tb_t)
        loss_phys = torch.mean(torch.sum(r_f**2, dim=1)) + torch.mean(torch.sum(r_s**2, dim=1)) + torch.mean(torch.sum(r_b**2, dim=1))
        # Partial physical boundary conditions, plus matching of full state.
        loss_bc = (
            torch.sum((pf[0, [2,3,6]] - left_partial)**2) +
            torch.sum((pb[0, [2,3,6]] - right_partial)**2) +
            (pf[-1, 0] - V)**2 + pf[-1, 1]**2 + pf[-1, 6]**2 +
            ps[-1, 0]**2 + ps[-1, 1]**2 + (ps[-1, 6] - 1.0)**2
        )
        loss_match = torch.sum((pf[-1] - ps[0])**2) + torch.sum((ps[-1] - pb[-1])**2)
        loss = phys_w*loss_phys + bc_w*loss_bc + match_w*loss_match
        loss.backward(); opt.step()
        loss_history.append(float(loss.detach().cpu()))
        if ep % PRINT_EVERY == 0 or ep == epochs:
            print(f"[{name} eps={eps:.1e}] epoch {ep:6d}/{epochs}, loss={loss_history[-1]:.3e}")

    pf = mf(t_t); ps = ms(tau_t); pb = mb(tb_t)
    r_f = fast_residual(pf, t_t)[:, :7]
    r_s = slow_residual(ps, tau_t)[:, :8]  # include electroneutrality as residual component
    r_b = fast_residual(pb, tb_t)[:, :7]
    pred_f = detach_np(pf); pred_s = detach_np(ps); pred_b = detach_np(pb)
    res_f = detach_np(r_f); res_s = detach_np(r_s); res_b = detach_np(r_b)

    segs = [
        SegmentResult("fast", t, pred_f, res_f, None),
        SegmentResult("slow", tau, pred_s, res_s, None),
        SegmentResult("fast2", tb, pred_b, res_b, None),
    ]
    matching = [endpoint_defect(pred_f[-1], pred_s[0]), endpoint_defect(pred_s[-1], pred_b[-1])]
    boundary = [
        endpoint_defect(pred_f[0, [2,3,6]], np.array([l1, l2, 0.0])),
        endpoint_defect(pred_b[0, [2,3,6]], np.array([r1, r2, 1.0])),
        abs(float(pred_f[-1, 0] - V)) + abs(float(pred_f[-1, 1])) + abs(float(pred_f[-1, 6])),
        abs(float(pred_s[-1, 0])) + abs(float(pred_s[-1, 1])) + abs(float(pred_s[-1, 6] - 1.0)),
    ]
    cert = compute_certificate(name, eps, segs, matching, boundary, loss_history[-1], epochs, n_points, time.time() - start_time)
    if SAVE_MODELS:
        torch.save({"fast": mf.state_dict(), "slow": ms.state_dict(), "fast2": mb.state_dict()}, MODEL_DIR / f"{name}_eps_{eps:.0e}.pt")
    return segs, cert, loss_history


TRAINERS = {
    "reaction_diffusion": train_reaction_diffusion,
    "lorenz": train_lorenz,
    "nonhyperbolic": train_nonhyperbolic,
    "pnp": train_pnp,
}

# =============================================================================
# 3. Main run and summary plots
# =============================================================================

def make_summary_figures(summary: pd.DataFrame) -> None:
    if summary.empty:
        return
    # Residual-to-error calibration, only rows with reference errors.
    df = summary[np.isfinite(summary["error_linf_sum"])].copy()
    if not df.empty:
        plt.figure(figsize=(6.2, 4.2))
        for b, g in df.groupby("benchmark"):
            plt.loglog(g["eta_l1"], g["error_linf_sum"], marker="o", linestyle="", label=b.replace("_", " "))
        xs = np.array([max(df["eta_l1"].min(), 1e-14), max(df["eta_l1"].max(), 1e-13)])
        plt.loglog(xs, xs, "--", label="reference E=eta")
        plt.xlabel(r"validation indicator $\eta_{val}$")
        plt.ylabel(r"reference error $E_{ref}$")
        plt.title("Residual-to-error certification")
        plt.grid(True, which="both", alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(FIG_DIR / "summary_residual_to_error.png", dpi=240)
        plt.close()

        plt.figure(figsize=(6.2, 4.2))
        for b, g in df.groupby("benchmark"):
            plt.semilogx(g["eps"], g["stability_ratio_eps_eta"], marker="o", label=b.replace("_", " "))
        plt.xlabel(r"$\varepsilon$")
        plt.ylabel(r"$E_{ref}/(\varepsilon+\eta_{val})$")
        plt.title("Empirical stability ratio")
        plt.grid(True, which="both", alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(FIG_DIR / "summary_stability_ratio.png", dpi=240)
        plt.close()

    plt.figure(figsize=(7.0, 4.2))
    for b, g in summary.groupby("benchmark"):
        plt.semilogy(g["eps"], g["eta_l1"], marker="o", label=b.replace("_", " "))
    plt.xlabel(r"$\varepsilon$")
    plt.ylabel(r"$\eta_{val}$")
    plt.title("Validation residual and defect indicator")
    plt.grid(True, which="both", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "summary_eta_validation.png", dpi=240)
    plt.close()


def main() -> None:
    print("="*78)
    print("GSPINN Project 1 one-file runner")
    print(f"RUN_MODE  : {RUN_MODE}")
    print(f"DEVICE    : {DEVICE}")
    print(f"DTYPE     : {DTYPE}")
    print(f"OUT_DIR   : {OUT_DIR}")
    print("="*78)

    rows = []
    for benchmark in BENCHMARKS:
        if benchmark not in TRAINERS:
            raise ValueError(f"Unknown benchmark: {benchmark}")
        for eps in EPS_VALUES:
            print("\n" + "-"*78)
            print(f"Running {benchmark}, eps={eps:.3e}")
            print("-"*78)
            trainer = TRAINERS[benchmark]
            segs, cert, loss_history = trainer(eps, EPOCHS[benchmark], N_POINTS)
            save_run_outputs(benchmark, eps, segs, cert, loss_history)
            row = asdict(cert)
            rows.append(row)
            print("Certificate:")
            for k, v in row.items():
                if isinstance(v, float):
                    print(f"  {k:28s}: {v:.6e}")
                else:
                    print(f"  {k:28s}: {v}")

    summary = pd.DataFrame(rows)
    summary_path = CSV_DIR / "project1_certification_summary.csv"
    summary.to_csv(summary_path, index=False)
    make_summary_figures(summary)

    print("\n" + "="*78)
    print("Finished.")
    print(f"Summary CSV: {summary_path}")
    print(f"Figures    : {FIG_DIR}")
    print(f"Trajectories/certificates: {OUT_DIR}")
    print("="*78)


if __name__ == "__main__":
    main()
    # In some notebook/cloud containers BLAS or plotting helper threads can keep
    # a plain Python script alive after all files are written.  We force a clean
    # script exit here.  Do not use this line if you paste the code into an
    # interactive notebook cell; it is intended for running this file as a script.
    import sys as _sys, os as _os
    _sys.stdout.flush()
    _sys.stderr.flush()
    _os._exit(0)

GSPINN Project 1 one-file runner
RUN_MODE  : paper
DEVICE    : cpu
DTYPE     : torch.float64
OUT_DIR   : /kaggle/working/gspinn_project1_outputs

------------------------------------------------------------------------------
Running reaction_diffusion, eps=1.000e-05
------------------------------------------------------------------------------
[reaction_diffusion eps=1.0e-05] epoch      0/20000, loss=6.067e+02
[reaction_diffusion eps=1.0e-05] epoch   2000/20000, loss=5.367e-01
[reaction_diffusion eps=1.0e-05] epoch   4000/20000, loss=2.012e-01
[reaction_diffusion eps=1.0e-05] epoch   6000/20000, loss=8.834e-02
[reaction_diffusion eps=1.0e-05] epoch   8000/20000, loss=1.928e-02
[reaction_diffusion eps=1.0e-05] epoch  10000/20000, loss=6.941e-03
[reaction_diffusion eps=1.0e-05] epoch  12000/20000, loss=3.481e-03
[reaction_diffusion eps=1.0e-05] epoch  14000/20000, loss=1.944e-03
[reaction_diffusion eps=1.0e-05] epoch  16000/20000, loss=1.191e-03
[reaction_diffusion eps=1.0e-05] epoch  18

KeyboardInterrupt: 